# HiRAG-Ontology: Multi-Agent Knowledge Graph Construction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ekaesha/hirag-ontology/blob/main/hirag_ontology_colab.ipynb)

**Bachelor's Thesis — HSE DSBA 2025**  
Authors: Eva Karimova, Alexey Popov

This notebook demonstrates:
- Loading a pre-built knowledge graph (2,314 entities, 2,346 relations)
- A3: Entity deduplication statistics
- A4: Consistency validation — Cons(G)
- Quality functional Q(G)
- Hybrid retrieval (TF-IDF + BM25 + PageRank)
- NetworkX knowledge graph visualisation

In [ ]:
# Cell 1: Install dependencies and clone repo
import subprocess, os, sys

subprocess.run(['pip', 'install', 'rank-bm25', 'networkx', 'matplotlib', '-q'], check=True)

REPO = '/content/hirag-ontology'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/ekaesha/hirag-ontology.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)

os.chdir(REPO)
sys.path.insert(0, REPO)
print(f'Working dir: {os.getcwd()}')
print('Setup complete!')

In [ ]:
# Cell 2: Download pre-built knowledge graph from GitHub
import json, urllib.request, os

os.makedirs('results', exist_ok=True)

GRAPH_URL = 'https://raw.githubusercontent.com/ekaesha/hirag-ontology/main/results/knowledge_graph_final.json'
GRAPH_PATH = 'results/knowledge_graph_final.json'

if not os.path.exists(GRAPH_PATH):
    print('Downloading knowledge graph from GitHub...')
    try:
        urllib.request.urlretrieve(GRAPH_URL, GRAPH_PATH)
        print('Downloaded!')
    except Exception as e:
        print(f'Download failed: {e}')
        print('Building minimal demo graph...')
        # Minimal demo graph with real data
        demo = {
            'entities': [
                {'id':'e1','label':'Острые лимфобластные лейкозы','entity_type':'Condition','description':'ALL','aliases':['ОЛЛ'],'source_chunks':[],'embedding':None},
                {'id':'e2','label':'Медикаментозная терапия','entity_type':'Procedure','description':'Drug therapy','aliases':[],'source_chunks':[],'embedding':None},
                {'id':'e3','label':'иматиниб','entity_type':'Drug','description':'BCR-ABL inhibitor','aliases':[],'source_chunks':[],'embedding':None},
                {'id':'e4','label':'Ph+ ОЛЛ','entity_type':'Condition','description':'Philadelphia-positive ALL','aliases':[],'source_chunks':[],'embedding':None},
                {'id':'e5','label':'FISH-исследование','entity_type':'LabTest','description':'FISH diagnostic test','aliases':[],'source_chunks':[],'embedding':None},
                {'id':'e6','label':'транслокация t(9;22)','entity_type':'Condition','description':'Philadelphia chromosome','aliases':[],'source_chunks':[],'embedding':None},
                {'id':'e7','label':'цитарабин','entity_type':'Drug','description':'Chemotherapy drug','aliases':[],'source_chunks':[],'embedding':None},
                {'id':'e8','label':'индукционная терапия','entity_type':'DosageRegimen','description':'Induction phase','aliases':[],'source_chunks':[],'embedding':None},
            ],
            'relations': [
                {'subject_id':'e2','predicate':'treats','object_id':'e1','confidence':0.95,'source_chunk':'demo'},
                {'subject_id':'e3','predicate':'treats','object_id':'e4','confidence':0.92,'source_chunk':'demo'},
                {'subject_id':'e4','predicate':'diagnosed_by','object_id':'e5','confidence':0.90,'source_chunk':'demo'},
                {'subject_id':'e4','predicate':'related_to','object_id':'e6','confidence':0.88,'source_chunk':'demo'},
                {'subject_id':'e7','predicate':'treats','object_id':'e1','confidence':0.85,'source_chunk':'demo'},
                {'subject_id':'e8','predicate':'treats','object_id':'e1','confidence':0.87,'source_chunk':'demo'},
            ]
        }
        with open(GRAPH_PATH, 'w', encoding='utf-8') as f:
            json.dump(demo, f, ensure_ascii=False, indent=2)
        print('Demo graph created.')

with open(GRAPH_PATH, encoding='utf-8') as f:
    kg_data = json.load(f)

print(f'Graph loaded: {len(kg_data["entities"])} entities, {len(kg_data["relations"])} relations')

In [ ]:
# Cell 3: Load into KnowledgeGraph object
from pipeline.knowledge_graph import KnowledgeGraph

kg = KnowledgeGraph.load(GRAPH_PATH)
print(f'KnowledgeGraph: {kg.stats()}')

In [ ]:
# Cell 4: A4 — Validation, compute Cons(G)
from pipeline.validator import ValidationAgent

validator = ValidationAgent()
val = validator.validate(kg)
print(f'Cons(G) = {val["consistency_score"]:.3f}')
print(f'Violations: {val["total_violations"]}')
print(f'By type: {val["violations_by_type"]}')

In [ ]:
# Cell 5: Compute Q(G)
from pipeline.quality import compute_quality

q = compute_quality(kg, val)
print(f'Q(G)         = {q["Q"]:.4f}')
print(f'Coverage     = {q["coverage"]:.3f}')
print(f'Consistency  = {q["consistency"]:.3f}')
print(f'Precision    = {q["precision"]:.3f}')
print(f'Redundancy   = {q["redundancy"]:.3f}')

In [ ]:
# Cell 6: Hybrid retrieval WITHOUT BERT — uses TF-IDF + BM25 + PageRank
import re, math
from collections import Counter
import networkx as nx

# Build NetworkX graph for PageRank
G = nx.DiGraph()
for eid in kg.entities:
    G.add_node(eid)
for r in kg.relations:
    G.add_edge(r.subject_id, r.object_id)
pagerank = nx.pagerank(G, alpha=0.85, max_iter=200)

# BM25 retrieval
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.sub(r'[^\w\s]', ' ', text.lower()).split()

entity_ids = list(kg.entities.keys())
corpus = []
for eid in entity_ids:
    e = kg.entities[eid]
    doc = e.label + ' ' + e.description + ' ' + ' '.join(e.aliases)
    corpus.append(tokenize(doc))

bm25 = BM25Okapi(corpus)

def rrf_fuse(lists, k=60):
    scores = {}
    for ranked in lists:
        for rank, (eid, _) in enumerate(ranked, 1):
            scores[eid] = scores.get(eid, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def retrieve(query, top_k=10):
    # BM25
    tokens = tokenize(query)
    bm25_scores = bm25.get_scores(tokens)
    bm25_ranked = sorted(zip(entity_ids, bm25_scores), key=lambda x: x[1], reverse=True)[:200]
    # PageRank structural
    pr_ranked = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:200]
    # RRF fusion
    fused = rrf_fuse([bm25_ranked, pr_ranked])
    return [eid for eid, _ in fused[:top_k]]

query = 'What is the treatment for acute lymphoblastic leukemia?'
results = retrieve(query)

print(f'Query: {query}')
print(f'Retrieved {len(results)} entities:')
for eid in results[:8]:
    e = kg.entities.get(eid)
    if e:
        print(f'  - {e.label} ({e.entity_type})')

In [ ]:
# Cell 7: NetworkX graph visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

TYPE_COLORS = {
    'Condition':'#534AB7', 'Drug':'#D85A30', 'Procedure':'#1D9E75',
    'LabTest':'#185FA5', 'AnatomicalStructure':'#378ADD',
    'DosageRegimen':'#BA7517', 'Symptom':'#639922',
    'Organization':'#D4537E', 'Other':'#888780'
}

degrees = dict(G.degree())
n = min(25, len(G.nodes()))
top_nodes = [nd for nd, _ in sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:n]]
subG = G.subgraph(top_nodes).copy()
pos = nx.spring_layout(subG, k=2.5, seed=42)

node_colors = [TYPE_COLORS.get(kg.entities[nd].entity_type if nd in kg.entities else 'Other', '#888') for nd in subG]
node_sizes  = [degrees[nd] * 18 + 150 for nd in subG]
labels      = {nd: (kg.entities[nd].label[:12] if nd in kg.entities else '') for nd in subG}

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_facecolor('#FAFAFA')
nx.draw_networkx_edges(subG, pos, alpha=0.4, arrows=True, ax=ax)
nx.draw_networkx_nodes(subG, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_labels(subG, pos, labels, font_size=7,
                        font_color='white', font_weight='bold', ax=ax)

legend = [mpatches.Patch(color=c, label=t) for t, c in TYPE_COLORS.items()
          if any((kg.entities[nd].entity_type if nd in kg.entities else 'Other') == t for nd in subG)]
ax.legend(handles=legend, loc='upper left', fontsize=8, framealpha=0.9)
ax.set_title(f'Knowledge Graph — Top {n} Entities by Degree\n'
             f'({len(kg.entities)} total entities · {len(kg.relations)} relations)',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('results/colab_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/colab_graph.png')

In [ ]:
# Cell 8: Summary
print('=' * 55)
print('HiRAG-Ontology — Results Summary')
print('=' * 55)
print(f'Entities:       {len(kg.entities):,}')
print(f'Relations:      {len(kg.relations):,}')
print(f'Cons(G):        {val["consistency_score"]:.3f}')
print(f'Q(G):           {q["Q"]:.4f}')
print(f'Top entity:     {max(degrees, key=degrees.get) and kg.entities.get(max(degrees, key=degrees.get), type("x", (), {"label": "?"})()).label}')
print('=' * 55)
print('RAG Evaluation (50 questions, 78 documents):')
print('  Naive RAG:           5.34')
print('  HiRAG baseline:      6.28')
print('  HiRAG-Ontology:      6.68  (+25.1% vs Naive RAG)')
print('=' * 55)